# Code to reproduce the results in the report

In [209]:
import pandas as pd

from discovery_child_development import PROJECT_DIR, S3_BUCKET, logging
from discovery_child_development.analysis.initial_results import utils
from nesta_ds_utils.loading_saving import S3

from discovery_child_development.utils import analysis_utils as au
from discovery_child_development.utils import plotting_utils as pu
from discovery_child_development.utils import chart_trends

# Remove altair warning
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

S3_OUTPUTS_DIR = '2024-07-iss-child-development/outputs/'

In [60]:
TECH = 'Technology'

# Taxonomy dataframe
topics_df = utils.load_topic_data()

# List all tech major categories
tech_subtypes = set(topics_df.query("type == @TECH").subtype.unique())
print(tech_subtypes)

{'Mobile', 'Immersive tech', 'Internet', 'AI'}


## Helper functions

In [210]:
# Relevant document ids for time series
def get_tech_ids(data_exploded_df):
    """Get relevant document ids for time series/growth estimations"""
    return (
        data_exploded_df
        .query("type == @TECH")
        .query("year >= 2013")
        .drop_duplicates('id')
        .id.to_list()
)

def get_tech_ids_5y(data_exploded_df: pd.DataFrame) -> pd.DataFrame:
    """Get relevant document ids for 2019-2023 stats"""
    return (
        data_exploded_df
        .query("type == @TECH")
        .query("year >= 2019")
        .drop_duplicates('id')
        .id.to_list()
    )

def report_magnitude_growth(magnitude_growth_df: pd.DataFrame) -> None:
    """Report magnitude and growth"""
    # Smoothed growth in 2019-2023
    growth = magnitude_growth_df.growth.iloc[0]
    # Total funding in 2019-2023 (in millions)
    magnitude = magnitude_growth_df.magnitude.iloc[0] * 5 / 1e+3

    logging.info(f"Growth in 2019-2023: {growth:.2f}%")
    logging.info(f"Total in 2019-2023: {magnitude:.2f}")

    return growth, magnitude

## Research funding
- Growth and total early-years digital tech research funding in 2019-2023
- Breakdowns by funder type (top funder; proportion from Innovate UK)
- Proportion of early-years funding associated with digital technologies
- Proportion and growth of major digital tech categories in 2019-2023
- Proportion and growth of major application areas in 2019-2023
- Baseline funding growth across all sectors in 2019-2023


In [211]:
# Load the data
ukri_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/data_ukri.csv',
        download_as='dataframe'
    )
    # Remove the instances tagged with expressive arts due to too much noise
    .query("topics != 'arts'")
)

# Explode by topics
ukri_exploded_df = utils.explode_data(ukri_df).query("topics != 'arts'")

# Relevant document ids for time series
# tech_ids = (
#     ukri_exploded_df
#     .query("type == @TECH")
#     .query("year >= 2013")
#     .drop_duplicates('id')
#     .id.to_list()
# )

# # Relevant document ids for 2019-2023 stats
# tech_ids_5y = (
#     ukri_exploded_df
#     .query("type == @TECH")
#     .query("year >= 2019")
#     .drop_duplicates('id')
#     .id.to_list()
# )

tech_ids = get_tech_ids(ukri_exploded_df)
tech_ids_5y = get_tech_ids_5y(ukri_exploded_df)


In [212]:
# Filter only technology projects
ukri_tech_type_df = (
    ukri_exploded_df
    .query("id in @tech_ids")
    .drop_duplicates(['id'])
)

ts_amounts_tech = utils.get_timeseries(ukri_tech_type_df, column='amount')
ts_counts_tech = utils.get_timeseries(ukri_tech_type_df, column='id')
utils.plot_quick_ts(ts_amounts_tech, 'amount')

alt.Chart(...)

### Growth and total early-years digital tech research funding

In [218]:
# Get magnitude and growth
magnitude_growth_ukri = au.ts_magnitude_growth_(
    ts_amounts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_ukri)

2024-07-23 18:38:20,994 - root - INFO - Growth in 2019-2023: 165.74%
2024-07-23 18:38:20,996 - root - INFO - Total in 2019-2023: 58.84


### Breakdowns by funder type
- Top funders
- Proportion from Innovate UK

In [219]:
# Check who are the top research funders for early-years digital tech
top_funders = (
    ukri_exploded_df
    .drop_duplicates(['id', 'lead_funder'])
    .query("id in @tech_ids_5y")    
    .groupby('lead_funder')
    .agg(amount=('amount', 'sum'))
    .sort_values('amount', ascending=False)
    .assign(proportion = lambda df: df.amount / df.amount.sum())
)
top_funders.head(5)

,amount,proportion
lead_funder,,
MRC,21617.556,0.367402
ESRC,5727.531,0.097343
Innovate UK,5584.073,0.094904
BBSRC,5425.603,0.092211
FLF,4700.880,0.079894


In [220]:
# Projects funded in the past five years
_funding_df = (
    ukri_exploded_df
    .query("id in @tech_ids_5y")
    .drop_duplicates('id')
)

# Get the total funding
funding_total = _funding_df.amount.sum()

# Get funding excluding health-related projects (for reference)
health_ids = ukri_exploded_df.query("type == 'Health'").id.to_list()
funding_wout_health = _funding_df.query("id not in @health_ids").amount.sum()

# Get Innovate UK funding specifically for with and without health
funding_innovate_uk = (
    _funding_df
    .query("lead_funder == 'Innovate UK'")
    .amount.sum())

funding_innovate_uk_wout_health = (
    _funding_df
    .query("lead_funder == 'Innovate UK'")
    .query("id not in @health_ids")
    .amount.sum())

# Calculate proportion of funding by Innovate UK
proportion_innovate_uk = funding_innovate_uk / funding_total
# Calculate proportion of funding by Innovate UK excluding health
proportion_innovate_uk_wout_health = funding_innovate_uk_wout_health / funding_wout_health

logging.info(f"Propotion of funding by Innovate UK: {proportion_innovate_uk:.2f}")
logging.info(f"Propotion of funding by Innovate UK excluding health: {proportion_innovate_uk_wout_health:.2f}")


2024-07-23 18:38:31,608 - root - INFO - Propotion of funding by Innovate UK: 0.09
2024-07-23 18:38:31,609 - root - INFO - Propotion of funding by Innovate UK excluding health: 0.17


### Proportion of early-years funding associated with digital technologies

In [221]:
# Calculate the total early-years project funding
total_early_years_funding = (
    ukri_exploded_df
    .query("year >= 2019 and year <= 2023")
    .drop_duplicates('id')
    .amount.sum())

# Get the proportion of funding for early-years digital tech
proportion_early_years_tech_ukri = funding_total / total_early_years_funding
logging.info(f"Proportion of funding for early-years digital tech: {proportion_early_years_tech_ukri:.2f}")

2024-07-23 18:38:36,497 - root - INFO - Proportion of funding for early-years digital tech: 0.20


In [222]:
# Check technology percentage using another approach
utils.get_data_distribution(
    ukri_exploded_df.query("year >= 2019 and year <= 2023"),
    column='type', 
    values=['id', 'amount']
)

,type,counts,counts_prop,amount,amount_prop
0,Biosciences,125,0.222,84393.908,0.288
1,Child care & preschool,23,0.041,16093.698,0.055
2,Development & learning,123,0.218,56456.251,0.193
3,General,348,0.618,195291.562,0.667
4,Health,362,0.643,209860.724,0.717
5,Parenting,20,0.036,8121.362,0.028
6,Society,133,0.236,68456.721,0.234
7,Technology,105,0.187,58838.909,0.201


### Proportion and growth of major digital tech categories in 2019-2023

In [223]:
# Filter only technology projects (major categories)
ukri_tech_subtype_df = (
    ukri_exploded_df
    .query('id in @tech_ids')
    .query("type == @TECH")
    .drop_duplicates(['id', 'subtype'])
)

ukri_tech_subtype_dist = (
    ukri_tech_subtype_df
    .query("year >= 2019 and year <= 2023")
    .groupby('subtype')
    .agg(
        counts=('id', 'nunique'), 
        amount=('amount', 'sum')
    )
    .reset_index()
    .assign(amount_prop = lambda df: round(df.amount / funding_total, 3))
)


In [224]:
column = 'subtype'
value = 'amount'

tech_subtype_ts = (
    ukri_tech_subtype_df
    .drop_duplicates(['id', column])
    .groupby(['subtype', 'year'])
    .agg(
        counts=('id', 'nunique'), 
        amount=('amount', 'sum')
    )
    .reset_index()
)

tech_subtype_ts = utils.impute_empty_periods_all_ts(tech_subtype_ts, column)

ukri_tech_magnitude_growth =utils.magnitude_and_growth(tech_subtype_ts, column, value)

In [164]:
ukri_tech_magnitude_growth

,magnitude,growth,subtype
0,8182.1622,147.968645,AI
0,1543.7518,23.475780,Immersive tech
0,1162.9148,378.897783,Internet
0,3573.8534,87.113273,Mobile


### Proportion and growth of major application areas

In [181]:
column = 'type'
hide_categories = ['Technology', 'General', 'Biosciences']

tech_applications_df = utils.get_data_distribution(
    ukri_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id', 'amount']
)

tech_applications_ts = utils.get_data_distribution(
    ukri_exploded_df
    .query('id in @tech_ids')
    # .query("type != 'Technology' and type != 'General'"),
    .query("type not in @hide_categories"),
    column=column, 
    values=['id', 'amount'],
    ts=True
)

trends_df = utils.get_data_magnitude_growth(
    ukri_exploded_df, ids=tech_ids, 
    column=column, 
    value='amount')


ukri_application_stats = (
        tech_applications_df
        .merge(trends_df.drop('counts', axis=1), on='type')[['type', 'magnitude', 'growth', 'counts', 'counts_prop', 'amount', 'amount_prop']]
        .query("type not in @hide_categories")
    )

In [179]:
ukri_application_stats

,type,magnitude,growth,counts,counts_prop,amount,amount_prop
1,Child care & preschool,518.0430,299.955036,8,0.076,2590.215,0.044
2,Development & learning,2528.1820,-51.255006,31,0.295,12640.91,0.215
4,Health,8050.1558,210.796765,64,0.61,40250.779,0.684
5,Parenting,611.8658,2541.467658,9,0.086,3059.329,0.052
6,Society,3251.1750,148.675879,21,0.2,16255.875,0.276


### Baseline funding growth across all sectors

In [189]:
gtr_df = S3.download_obj(
    bucket = S3_BUCKET,
    path_from = S3_OUTPUTS_DIR + 'gtr_texts.csv',
    download_as='dataframe'
)

In [207]:
ukri_baseline_df = utils.get_baseline_ukri(gtr_df)

trends_baseline = au.ts_magnitude_growth_(
    ts_df = ukri_baseline_df,
    year_start = 2019,
    year_end = 2023  
)
ukri_baseline_magnitude = trends_baseline.loc['amount'].magnitude
ukri_baseline_growth = trends_baseline.loc['amount'].growth
logging.info(f"UKRI baseline growth: {ukri_baseline_growth:.2f}%")

2024-07-23 18:32:17,857 - root - INFO - UKRI baseline growth: -5.05%


## Research publications